# Example 1: Grid World Navigation

Here we have an agent moving in a simple 1D grid world.  The agent can be in one of four positions:
- "left"
- "center_left"
- "center_right"
- "right"

and can take only two actions:
- "move_right"
- "move_left"


In [1]:
import jax.tree_util as jtu
from jax import numpy as jnp
from jax import random as jr
from pymdp.agent import Agent
from pymdp.distribution import compile_model
from pymdp.envs.env import Env
from pymdp.envs import rollout

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [2]:
# Define the labels for the agent's position on the 1D grid
positions = ["left", "center_left", "center_right", "right"]
# Define the labels for the agent's available actions
actions = ["move_left", "move_right"]

# The model description is specified by a nested dictionary
model_description = {
    "observations": {
        "position_obs": {
            "elements": positions, 
            "depends_on": ["position"] # we specify that the observation depends on the "position" state factor
        },
    },
    "controls": {
        "movement": {"elements": actions} # we specify the available actions
    },
    "states": {
        "position": {
            "elements": positions, 
            "depends_on": ["position"],  # our current position depends on previous position...
            "controlled_by": ["movement"]  # ...and the movement action taken
        },
    },
}

# compile the model structure from the description
model = compile_model(model_description)

We have now built a generative model structure using the model description. However the model's parameters (e.g., A, B, D, etc.) are currently uninitialized arrays of zeros. So now, we can fill in the parameter tensors by using the axis and element labels we defined in the model description dict, to set values in particular indices of these arrays.

In [3]:
# Fill in the likelihood (A) tensor
# The observations have an identical mapping to the states 
# (i.e., the agent will perfectly observe its position)
model.A["position_obs"]["left", "left"] = 1.0
model.A["position_obs"]["center_left", "center_left"] = 1.0
model.A["position_obs"]["center_right", "center_right"] = 1.0
model.A["position_obs"]["right", "right"] = 1.0
# You could also use the .data attribute to set the identity mapping directly:
# model.A["position_obs"].data = jnp.eye(len(positions)) 

# Fill in the transition model (B) tensor
# note that it's specified as ["to", "from", "action"]

# moving right
model.B["position"]["center_left", "left", "move_right"] = 1.0     
model.B["position"]["center_right", "center_left", "move_right"] = 1.0  
model.B["position"]["right", "center_right", "move_right"] = 1.0    
model.B["position"]["right", "right", "move_right"] = 1.0           

# moving left  
model.B["position"]["left", "left", "move_left"] = 1.0              
model.B["position"]["left", "center_left", "move_left"] = 1.0       
model.B["position"]["center_left", "center_right", "move_left"] = 1.0  
model.B["position"]["center_right", "right", "move_left"] = 1.0    

# set preferences (C) tensor - prefer to be at "center_left"
model.C["position_obs"]["center_left"] = 1.0

Now, let's create the agent (via the Agent object) and have it infer which state it is in via an observation and select an action according to its goal.

In [4]:
# The parameter gamma defines the degree of stocasticity in behavior.
# We choose here deterministic behavior; make gamma smaller for stochastic behavior
gamma = 10 

# Create agent
agent = Agent(**model, gamma = gamma)

# Set up initial observation to be "left":
# broadcast to agent's batch size (defaults to 1 agent) and add a time dimension
observation = jnp.zeros((agent.batch_size, 1)) 

# Get the prior:
# qs needs a time dimension too
qs_init = jtu.tree_map(lambda x: jnp.expand_dims(x, 1), agent.D) 

# print initial beliefs, goal, and action chosen
qs = agent.infer_states([observation], qs_init)
print(f"Current belief about position: {positions[jnp.argmax(qs[0][0])]}")
qs = [jnp.squeeze(q, 1) for q in qs]

print(f"Goal position: {positions[jnp.argmax(agent.C[0])]}")

q_pi, G = agent.infer_policies(qs)
action_idx = agent.sample_action(q_pi)
print(f"Action chosen: {actions[action_idx[0][0]]}")

<positron-console-cell-4>:6: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.


Current belief about position: left
Goal position: center_left
Action chosen: move_right


We can also run multiple agents or independent trials in parallel, each with a different initial observation (i.e., different initial position) by using the `batch_size` argument:

In [5]:
batch_size = 3 # running 3 trials or agents in parallel
gamma = 10     # deterministic behavior; make gamma smaller for stochastic behavior

# create agent
agent = Agent(**model, batch_size=batch_size, gamma=gamma)

# set up different initial observations for each agent: "left", "center_right", and "right"
observation = [jnp.array([[0], [2], [3]])] # wrap in a list to indicate the single modality; observation[0].shape = (batch_size, 1)

# get the prior
qs_init = jtu.tree_map(lambda x: jnp.expand_dims(x, 1), agent.D) # qs needs a time dimension too

# print goal and initial beliefs
qs = agent.infer_states(observation, qs_init)
for a in range(batch_size): 
    print(f"Agent {a}'s current belief about position: {positions[jnp.argmax(qs[0][a])]}")
qs = [jnp.squeeze(q, 1) for q in qs]

print(f"\nGoal position for all agents: {positions[jnp.argmax(agent.C[0])]}\n")

q_pi, G = agent.infer_policies(qs)
action = agent.sample_action(q_pi)
for a in range(batch_size): 
    print(f"Agent {a}'s action chosen: {actions[action[a][0]]}")

<positron-console-cell-5>:5: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.


Agent 0's current belief about position: left
Agent 1's current belief about position: center_right
Agent 2's current belief about position: right

Goal position for all agents: center_left

Agent 0's action chosen: move_right
Agent 1's action chosen: move_left
Agent 2's action chosen: move_left
